# Data Cleaning

## 1. Goal
Build a reproducible cleaning pipeline and generate a Data Quality Report.

## 2. Imports

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
project_root = Path(os.getcwd()).parent.parent
sys.path.append(str(project_root))

from notebooks.config import *
from notebooks.utils.notebook_helpers import set_seed
set_seed(GLOBAL_RANDOM_SEED)

## 3. Data Loading

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Initial shape: {df.shape}")
df.head()

Initial shape: (8173, 46)


,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


## 4. Data Quality Report - Initial

In [3]:
# Missing Values Summary
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).sort_values('Missing %', ascending=False)
print("--- Missing Values Report ---")
display(missing_df[missing_df['Missing Count'] > 0])

--- Missing Values Report ---


,Missing Count,Missing %
meta_data,8173,100.000000
comment,8173,100.000000
map_file,8173,100.000000
direction,8130,99.473877
resolved_at_longitude,8099,99.094580
resolved_at_latitude,8099,99.094580
resolved_at_address,8099,99.094580
resolved_by_id,8099,99.094580
resolved_datetime,8099,99.094580
citizen_accident_id,8045,98.433868


In [4]:
# Invalid Coordinate Report
invalid_coords = df[
    (df['latitude'] < -90) | (df['latitude'] > 90) | 
    (df['longitude'] < -180) | (df['longitude'] > 180)
]
print(f"Found {len(invalid_coords)} rows with invalid coordinates out of bounds (-90 to 90, -180 to 180).")

Found 0 rows with invalid coordinates out of bounds (-90 to 90, -180 to 180).


## 5. Cleaning Operations
We will not aggressively drop columns yet, but we will clean the ones we strictly need.

In [5]:
# Parse Dates
date_cols = ['start_datetime', 'resolved_datetime', 'closed_datetime', 'modified_datetime']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
        
# Check timestamp inconsistencies
invalid_times = df[df['resolved_datetime'] < df['start_datetime']]
print(f"Found {len(invalid_times)} rows where resolved_datetime is before start_datetime.")


Found 0 rows where resolved_datetime is before start_datetime.


In [6]:
# Drop explicitly broken rows or handle them safely
# We will drop rows where start_datetime is NaT as we cannot do operational analysis without a start time.
initial_len = len(df)
df = df.dropna(subset=['start_datetime'])

# Drop rows with invalid coordinates
if len(invalid_coords) > 0:
    df = df.drop(invalid_coords.index)
    
# Fix invalid times by swapping or dropping. Here we drop for simplicity.
if len(invalid_times) > 0:
    df = df.drop(invalid_times.index)

print(f"Dropped {initial_len - len(df)} rows due to fundamental data corruption.")

Dropped 0 rows due to fundamental data corruption.


In [7]:
# Fill essential missing values with 'unknown' or median
fill_cols = ['event_cause', 'veh_type', 'priority']
for col in fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

## 6. Export Intermediate Cleaned Data

In [8]:
# Create intermediate dir if not exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
intermediate_path = PROCESSED_DATA_DIR / "intermediate_cleaned.parquet"
df.to_parquet(intermediate_path, index=False)
print(f"Saved {len(df)} cleaned rows to {intermediate_path}")

Saved 8173 cleaned rows to C:\Users\akash\gridwise-ai\data\processed\intermediate_cleaned.parquet
